# Multilingual LVC dataset

## French

This is the version currently in use

### active sentences

In [ ]:
import pandas as pd

file1 = "collocation_items_FR_v1_utf8.csv"
file2 = "sentence_buildersFR2_utf8.csv"
file3 = "french_verbs_inflection_sub_utf8.csv"

items = pd.read_csv(file1, sep=None, engine="python", encoding="utf8")
builder = pd.read_csv(file2, sep=None, engine="python", encoding="utf8")
verbs = pd.read_csv(file3, sep=None, engine="python", encoding="utf8")

for df in [items, builder, verbs]:
    df.columns = df.columns.str.strip()
    df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], inplace=True)
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()

time_specs = builder["time_specs"].replace("nan", pd.NA).dropna()
time_specs = time_specs[time_specs != ""].unique().tolist()

adverbs = builder["adverbs"].replace("nan", pd.NA).dropna()
adverbs = adverbs[adverbs != ""].unique().tolist()

subjects = ["ils"]

tense_columns = {
    "future": "future_active",
    "present_perfect": "present_perfect_active",
    "past_perfect": "past_perfect_active",
}

output_rows = {
    "future_no_adverbs": [],
    "future_with_adverbs": [],
    "present_perfect_no_adverbs": [],
    "present_perfect_with_adverbs": [],
    "past_perfect_no_adverbs": [],
    "past_perfect_with_adverbs": [],
}

def get_verb_form(verb, subject, tense_col):
    row = verbs[(verbs["verb"] == verb) & (verbs["person"] == subject)].iloc[0]
    return row[tense_col]

def get_object_phrase(full_expression, verb):
    return full_expression.replace(verb + " ", "", 1)

def add_adverb(verb_form, adverb, tense):
    if tense == "future":
        return f"{verb_form} {adverb}"
    aux, participle = verb_form.split(" ", 1)
    return f"{aux} {adverb} {participle}"

for _, item in items.iterrows():

    verb = item["verb_fr"]
    obj = get_object_phrase(item["full_expression"], verb)

    for time_spec in time_specs:
        for subject in subjects:
            for tense, tense_col in tense_columns.items():

                verb_form = get_verb_form(verb, subject, tense_col)

                sentence = f"{time_spec}, {subject} {verb_form} {obj}."

                row = item.to_dict()
                row.update({
                    "sentence": sentence,
                    "voice": "active",
                    "tense": tense,
                    "adverb": "NA",
                    "subject": subject,
                    "time_spec": time_spec
                })
                output_rows[f"{tense}_no_adverbs"].append(row)

                for adverb in adverbs:
                    verb_form_adv = add_adverb(verb_form, adverb, tense)
                    sentence = f"{time_spec}, {subject} {verb_form_adv} {obj}."

                    row = item.to_dict()
                    row.update({
                        "sentence": sentence,
                        "voice": "active",
                        "tense": tense,
                        "adverb": adverb,
                        "subject": subject,
                        "time_spec": time_spec
                    })
                    output_rows[f"{tense}_with_adverbs"].append(row)

for name, rows in output_rows.items():

    out = pd.DataFrame(rows)
    out = out.dropna(axis=1, how="all")

    filename = f"LVC_FR_active_{name}.csv"
    out.to_csv(filename, index=False, encoding="utf-8-sig")

    print(f"Saved {len(out)} rows to {filename}")

### passive

In [ ]:
import pandas as pd

file1 = "collocation_items_FR_v1_utf8.csv"
file2 = "sentence_buildersFR2_utf8.csv"
file3 = "french_verbs_inflection_sub_utf8.csv"

items = pd.read_csv(file1, sep=None, engine="python", encoding="utf8")
builder = pd.read_csv(file2, sep=None, engine="python", encoding="utf8")
verbs = pd.read_csv(file3, sep=None, engine="python", encoding="utf8")

for df in [items, builder, verbs]:
    df.columns = df.columns.str.strip()
    df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], inplace=True)
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()

time_specs = builder["time_specs"].replace("nan", pd.NA).dropna()
time_specs = time_specs[time_specs != ""].unique().tolist()

adverbs = builder["adverbs"].replace("nan", pd.NA).dropna()
adverbs = adverbs[adverbs != ""].unique().tolist()

tense_columns = {
    "future": "future_passive",
    "present_perfect": "present_perfect_passive",
    "past_perfect": "past_perfect_passive",
}

output_rows = {
    "future_no_adverbs": [],
    "future_with_adverbs": [],
    "present_perfect_no_adverbs": [],
    "present_perfect_with_adverbs": [],
    "past_perfect_no_adverbs": [],
    "past_perfect_with_adverbs": [],
}

def get_object_phrase(full_expression, verb):
    return full_expression.replace(verb + " ", "", 1)

def infer_passive_person(obj):
    """
    Chooses the verb-table person needed for passive agreement.

    il   = masculine singular
    elle = feminine singular
    ils  = masculine plural / mixed plural
    elles = feminine plural

    This is intentionally simple and determiner-based.
    """
    obj = obj.lower().strip()

    if obj.startswith(("les ", "des ")):
        return "ils"

    if obj.startswith(("une ", "la ", "de la ")):
        return "elle"

    if obj.startswith(("un ", "le ", "du ")):
        return "il"

    # l' forms are ambiguous; default to masculine singular
    return "il"

def get_verb_form(verb, passive_person, tense_col):
    row = verbs[
        (verbs["verb"] == verb) &
        (verbs["person"] == passive_person)
    ].iloc[0]
    return row[tense_col]

def add_adverb(passive_form, adverb, tense):
    if tense == "future":
        aux, participle = passive_form.split(" ", 1)
        return f"{aux} {adverb} {participle}"

    aux, rest = passive_form.split(" ", 1)
    return f"{aux} {adverb} {rest}"

for _, item in items.iterrows():

    verb = item["verb_fr"]
    obj = get_object_phrase(item["full_expression"], verb)

    passive_person = infer_passive_person(obj)

    for time_spec in time_specs:

        for tense, tense_col in tense_columns.items():

            verb_form = get_verb_form(verb, passive_person, tense_col)

            sentence = f"{time_spec}, {obj} {verb_form}."

            row = item.to_dict()
            row.update({
                "sentence": sentence,
                "voice": "passive",
                "tense": tense,
                "adverb": "NA",
                "subject": obj,
                "time_spec": time_spec
            })

            output_rows[f"{tense}_no_adverbs"].append(row)

            for adverb in adverbs:

                verb_form_adv = add_adverb(verb_form, adverb, tense)
                sentence = f"{time_spec}, {obj} {verb_form_adv}."

                row = item.to_dict()
                row.update({
                    "sentence": sentence,
                    "voice": "passive",
                    "tense": tense,
                    "adverb": adverb,
                    "subject": obj,
                    "time_spec": time_spec
                })

                output_rows[f"{tense}_with_adverbs"].append(row)

for name, rows in output_rows.items():

    out = pd.DataFrame(rows)
    out = out.dropna(axis=1, how="all")

    filename = f"LVC_FR_passive_{name}.csv"
    out.to_csv(filename, index=False, encoding="utf-8-sig")

    print(f"Saved {len(out)} rows to {filename}")

### nominal subjects

In [ ]:
import pandas as pd
import random

# =========================
# SETTINGS
# =========================

SUBJECT_NUMBER = "both"
# "sg", "pl", "both"

SUBJECT_DET_TYPES = ["det"]
# "det", "det_poss", "no_det"

USE_PRONOUN_SUBJECTS = True
USE_NOMINAL_SUBJECTS = True

USE_ADVERBS = True
TENSES = ["present_perfect"]
# "future", "present_perfect", "past_perfect"

RANDOM_SUBJECT_SELECTION = True
RANDOM_OUTPUT_SAMPLE = True
REMOVE_DUPLICATE_SENTENCES = True

N_RANDOM_ROWS = 1000
RANDOM_STATE = 99

SPLIT_OUTPUT = False
ROWS_PER_FILE = 10000

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_FR_v1_utf8.csv"
file2 = "sentence_buildersFR2_utf8.csv"
file3 = "french_verbs_inflection_sub_utf8.csv"

output_file = "LVC_FR_active_subjects_output.csv"

random.seed(RANDOM_STATE)

# =========================
# READ FILES
# =========================

items = pd.read_csv(file1, sep=None, engine="python", encoding="latin1")
builder = pd.read_csv(file2, sep=None, engine="python", encoding="latin1")
verbs = pd.read_csv(file3, sep=None, engine="python", encoding="latin1")

for df in [items, builder, verbs]:
    df.columns = (
        df.columns
        .str.strip()
        .str.replace("\n", "", regex=False)
        .str.replace("\r", "", regex=False)
    )
    df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], inplace=True)
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()

print("Builder columns:")
print(builder.columns.tolist())

# =========================
# HELPERS
# =========================

def clean_list(series):
    values = (
        series.astype(str)
        .str.strip()
        .replace("nan", pd.NA)
        .dropna()
    )
    values = values[values != ""]
    return values.unique().tolist()

def get_col(row, possible_names):
    for name in possible_names:
        if name in row.index:
            return str(row.get(name, "")).strip()
    return ""

def starts_with_vowel_or_h(word):
    return str(word).lower().startswith((
        "a", "à", "â",
        "e", "é", "è", "ê", "ë",
        "i", "î", "ï",
        "o", "ô",
        "u", "ù", "û", "ü",
        "y",
        "h"
    ))

def make_np(det, noun):
    if det == "NA":
        return noun
    if det.endswith("'"):
        return f"{det}{noun}"
    return f"{det} {noun}"

def subject_prefix(subject, verb_form):
    if subject == "je" and starts_with_vowel_or_h(verb_form):
        return "j'"
    return subject + " "

def get_object_phrase(full_expression, verb):
    return full_expression.replace(verb + " ", "", 1)

def get_verb_form(verb, person, tense_col):
    match = verbs[
        (verbs["verb"] == verb) &
        (verbs["person"] == person)
    ]

    if match.empty:
        raise ValueError(
            f"No verb form for verb={verb}, person={person}, tense={tense_col}"
        )

    return match.iloc[0][tense_col]

def add_adverb(verb_form, adverb, tense):
    if adverb == "NA":
        return verb_form

    if tense == "future":
        return f"{verb_form} {adverb}"

    aux, participle = verb_form.split(" ", 1)
    return f"{aux} {adverb} {participle}"

def compatible_singular_dets(dets, noun):
    vowel = starts_with_vowel_or_h(noun)
    output = []

    for det in dets:
        if det == "l'" and vowel:
            output.append(det)
        elif det in ["le", "la"] and vowel:
            continue
        elif det == "l'" and not vowel:
            continue
        else:
            output.append(det)

    return output

def choose_possessives_sg(context_gender, noun):
    if context_gender == "F":
        if starts_with_vowel_or_h(noun):
            return poss_fem_vowel
        return poss_fem
    return poss_masc

# =========================
# BASIC LISTS
# =========================

time_specs = clean_list(builder["time_specs"])
adverbs = clean_list(builder["adverbs"])

det_masc = clean_list(builder["det_masc"])
det_fem = clean_list(builder["det_fem"])
det_pl = clean_list(builder["det_pl"])

poss_masc = clean_list(builder["poss_masc"])
poss_fem = clean_list(builder["poss_fem"])
poss_fem_vowel = clean_list(builder["poss_fem_vowel"])
poss_pl = clean_list(builder["poss_pl"])

pronouns = clean_list(builder["subjects_pron"])

# =========================
# SUBJECT PHRASES
# =========================

subject_rows = []

# Pronoun subjects
if USE_PRONOUN_SUBJECTS:
    for pron in pronouns:
        subject_rows.append({
            "subject": pron,
            "subject_base": pron,
            "subject_number": "sg" if pron in ["je", "tu", "il", "elle", "on"] else "pl",
            "subject_det": "NA",
            "subject_det_type": "NA",
            "subject_type": "pronoun",
            "person": pron,
            "context_gender": "NA",
            "possessive": "NA"
        })

# Nominal subjects
if USE_NOMINAL_SUBJECTS:

    for _, row in builder.iterrows():

        subj_sg = get_col(row, ["subject_sg", "subjects_sg"])
        subj_pl = get_col(row, ["subject_pl", "subjects_pl"])

        context_gender = get_col(row, ["context_gender"])
        possessive = get_col(row, ["possessive"])

        # -------------------------
        # Singular nominal subjects
        # -------------------------

        if subj_sg not in ["", "nan"]:

            person = "elle" if context_gender == "F" else "il"

            dets_sg = det_fem if context_gender == "F" else det_masc
            dets_sg = compatible_singular_dets(dets_sg, subj_sg)

            for det in dets_sg:
                subject_rows.append({
                    "subject": make_np(det, subj_sg),
                    "subject_base": subj_sg,
                    "subject_number": "sg",
                    "subject_det": det,
                    "subject_det_type": "det",
                    "subject_type": "nominal",
                    "person": person,
                    "context_gender": context_gender,
                    "possessive": possessive
                })

            if possessive == "T":
                for det in choose_possessives_sg(context_gender, subj_sg):
                    subject_rows.append({
                        "subject": make_np(det, subj_sg),
                        "subject_base": subj_sg,
                        "subject_number": "sg",
                        "subject_det": det,
                        "subject_det_type": "det_poss",
                        "subject_type": "nominal",
                        "person": person,
                        "context_gender": context_gender,
                        "possessive": possessive
                    })

        # -------------------------
        # Plural nominal subjects
        # -------------------------

        if subj_pl not in ["", "nan"]:

            person = "elles" if context_gender == "F" else "ils"

            subject_rows.append({
                "subject": subj_pl,
                "subject_base": subj_pl,
                "subject_number": "pl",
                "subject_det": "NA",
                "subject_det_type": "no_det",
                "subject_type": "nominal",
                "person": person,
                "context_gender": context_gender,
                "possessive": possessive
            })

            for det in det_pl:
                subject_rows.append({
                    "subject": make_np(det, subj_pl),
                    "subject_base": subj_pl,
                    "subject_number": "pl",
                    "subject_det": det,
                    "subject_det_type": "det",
                    "subject_type": "nominal",
                    "person": person,
                    "context_gender": context_gender,
                    "possessive": possessive
                })

            if possessive == "T":
                for det in poss_pl:
                    subject_rows.append({
                        "subject": make_np(det, subj_pl),
                        "subject_base": subj_pl,
                        "subject_number": "pl",
                        "subject_det": det,
                        "subject_det_type": "det_poss",
                        "subject_type": "nominal",
                        "person": person,
                        "context_gender": context_gender,
                        "possessive": possessive
                    })

subjects_df = pd.DataFrame(subject_rows).drop_duplicates()

# =========================
# FILTER SUBJECTS
# =========================

if SUBJECT_NUMBER != "both":
    subjects_df = subjects_df[subjects_df["subject_number"] == SUBJECT_NUMBER]

nominal_subjects = subjects_df[
    (subjects_df["subject_type"] == "nominal") &
    (subjects_df["subject_det_type"].isin(SUBJECT_DET_TYPES))
]

if USE_PRONOUN_SUBJECTS:
    pronoun_subjects = subjects_df[subjects_df["subject_type"] == "pronoun"]
    subjects_df = pd.concat([pronoun_subjects, nominal_subjects], ignore_index=True)
else:
    subjects_df = nominal_subjects

subjects_df = subjects_df.reset_index(drop=True)

print("Number of available subjects:", len(subjects_df))
print(subjects_df["subject"].head(50))
print(subjects_df["subject_type"].value_counts())

subject_records = subjects_df.to_dict("records")

# =========================
# TENSE COLUMNS
# =========================

tense_columns = {
    "future": "future_active",
    "present_perfect": "present_perfect_active",
    "past_perfect": "past_perfect_active",
}

# =========================
# BUILD SENTENCES
# =========================

rows = []

for _, item in items.iterrows():

    verb = item["verb_fr"]
    obj = get_object_phrase(item["full_expression"], verb)

    for time_spec in time_specs:

        adverb_options = adverbs if USE_ADVERBS else ["NA"]

        for adverb in adverb_options:
            for tense in TENSES:

                tense_col = tense_columns[tense]

                if RANDOM_SUBJECT_SELECTION:
                    subject_iterator = [random.choice(subject_records)]
                else:
                    subject_iterator = subject_records

                for subj_info in subject_iterator:

                    subject = subj_info["subject"]
                    person = subj_info["person"]

                    verb_form = get_verb_form(verb, person, tense_col)
                    verb_form = add_adverb(verb_form, adverb, tense)

                    sentence_body = (
                        f"{subject_prefix(subject, verb_form)}"
                        f"{verb_form} {obj}"
                    )

                    sentence = f"{time_spec}, {sentence_body}."

                    new_row = item.to_dict()
                    new_row.update(subj_info)
                    new_row.update({
                        "sentence": sentence,
                        "voice": "active",
                        "tense": tense,
                        "adverb": adverb,
                        "time_spec": time_spec
                    })

                    rows.append(new_row)

# =========================
# FINAL OUTPUT
# =========================

out = pd.DataFrame(rows)

if REMOVE_DUPLICATE_SENTENCES:
    out = out.drop_duplicates(subset=["sentence"])

if RANDOM_OUTPUT_SAMPLE:
    out = out.sample(
        n=min(N_RANDOM_ROWS, len(out)),
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

out = out.dropna(axis=1, how="all")

empty_cols = []

for col in out.columns:
    values = out[col].astype(str).str.strip().replace("nan", "")

    if (values == "").all():
        empty_cols.append(col)

out = out.drop(columns=empty_cols)

# =========================
# SAVE
# =========================

if not SPLIT_OUTPUT:

    out.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Saved {len(out)} rows to {output_file}")

else:

    n_files = (len(out) - 1) // ROWS_PER_FILE + 1

    for i in range(n_files):

        start = i * ROWS_PER_FILE
        end = min((i + 1) * ROWS_PER_FILE, len(out))

        chunk = out.iloc[start:end]

        filename = output_file.replace(
            ".csv",
            f"_part{i+1:03d}.csv"
        )

        chunk.to_csv(filename, index=False, encoding="utf-8-sig")

        print(f"Saved rows {start+1:,}-{end:,} to {filename}")

    print(f"\nFinished: {len(out):,} rows written to {n_files} files.")